# Training Risk & Yield Model — AgroSense

Notebook ini melatih **dua model** untuk platform AgroSense:

1. **Risk Model** — klasifikasi biner (gagal panen / tidak) dengan output `risk_percentage` (0-100) untuk pasangan farm + crop. Dibaca oleh Lambda `lambda_risk_prediction`.
2. **Yield Model** — regresi/time-series untuk forecast volume panen N periode ke depan per crop. Dibaca oleh Lambda `lambda_yield_forecasting`.

Input dibaca dari hasil ETL di S3 (`processed-data/farm_features` dan `processed-data/crop_stats`), dengan fallback ke dataset sintetis lokal bila data S3 belum tersedia (untuk development end-to-end).

## Konfigurasi

- `S3_BUCKET`: nama bucket S3 (isi sesuai environment, default placeholder `[BUCKET]`).
- Model di-serialisasi dengan `pickle` dan di-upload ke:
  - `s3://[BUCKET]/models/risk_model.pkl`
  - `s3://[BUCKET]/models/yield_model.pkl`

In [ ]:
import os
import pickle
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    root_mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---- Konfigurasi ----
S3_BUCKET = os.environ.get("S3_BUCKET", "[BUCKET]")
RAW_BASE = f"s3://{S3_BUCKET}/processed-data/"
MODEL_PREFIX = "models/"
np.random.seed(42)

try:
    import s3fs  # optional, untuk baca Parquet langsung dari S3
except ImportError:
    s3fs = None

print(f"S3 bucket: {S3_BUCKET}")
print("s3fs available:", s3fs is not None)

## Baca data hasil ETL dari S3 (dengan fallback lokal)

Jika `s3fs` tersedia dan `S3_BUCKET` bukan placeholder (`[`), baca `farm_features` dan `crop_stats`. Jika tidak (development tanpa AWS), gunakan dataset sintetis.

In [ ]:
def read_parquet_s3(prefix: str) -> pd.DataFrame | None:
    """Baca folder Parquet dari S3. Return None bila tidak bisa."""
    if s3fs is None or "[" in S3_BUCKET:
        return None
    try:
        fs = s3fs.S3FileSystem()
        files = fs.glob(f"{RAW_BASE}{prefix}/*.parquet")
        if not files:
            return None
        return pd.read_parquet(f"s3://{S3_BUCKET}/{prefix}/", filesystem=fs)
    except Exception as exc:  # noqa: BLE001
        print(f"[WARN] S3 read gagal ({exc}); pakai fallback lokal.")
        return None

farm_features = read_parquet_s3("processed-data/farm_features")
crop_stats = read_parquet_s3("processed-data/crop_stats")
print("farm_features from S3:", farm_features is not None)
print("crop_stats from S3:", crop_stats is not None)

## Dataset sintetis untuk training

Fungsi ini menghasilkan data farm-level berlabel (gagal panen atau tidak) plus data hasil panen per crop sebagai fallback end-to-end.

In [ ]:
def build_toy_sets(n_farms=1000, n_crops=500, n_rows=20000):
    """Buat dataset sintetis kecil: farm features + harvest history per crop."""
    farm_ids = np.array([f"F{i:05d}" for i in range(1, n_farms + 1)], dtype=object)
    crop_ids = np.array([f"C{i:05d}" for i in range(1, n_crops + 1)], dtype=object)

    n = n_farms
    farm_size = np.random.uniform(0.5, 50.0, size=n)
    total_activities = np.random.randint(1, 80, size=n)
    total_volume = total_activities * np.random.uniform(1, 60, size=n)
    avg_volume = total_volume / total_activities
    diversity = np.random.randint(1, 6, size=n)
    region_avg_quantity = np.random.uniform(20, 300, size=n)

    risk_score = (
        0.15 * (1 / np.sqrt(farm_size))
        + 0.3 * (total_activities / 80)
        + 0.2 * (diversity / 5)
        + np.random.normal(0, 0.15, size=n)
    )
    label = (risk_score > np.quantile(risk_score, 0.55)).astype(int)
    risk_pct = np.clip(risk_score / np.max(risk_score) * 100, 0, 100)

    farms_df = pd.DataFrame({
        "farm_id": farm_ids,
        "region": np.random.choice(["Jawa Tengah", "Jawa Barat", "Sumatera Utara"], size=n),
        "soil_type": np.random.choice(["Andosol", "Aluvial", "Gambut"], size=n),
        "farmer_segment": np.random.choice(["subsisten", "komersial", "koperasi"], size=n),
        "irrigation_type": np.random.choice(["tadah_hujan", "irigasi_teknis", "pompa"], size=n),
        "farm_size_hectare": farm_size,
        "total_activities": total_activities,
        "total_activity_volume": total_volume,
        "avg_activity_volume": avg_volume,
        "crop_diversity": diversity,
        "region_avg_quantity": region_avg_quantity,
        "risk_label": label,
        "risk_percentage": risk_pct,
    })

    area = np.random.uniform(1, 80, size=n_rows)
    base_yield = np.random.uniform(1.2, 6.0, size=n_rows)
    seasons = [f"season_{y}-{s}" for y in range(2018, 2027) for s in (1, 2)]
    crop_col = crop_ids[np.random.randint(0, n_crops, size=n_rows)]
    qty = area * base_yield
    rows = []
    for i in range(n_rows):
        rows.append({
            "crop_id": crop_col[i],
            "season": np.random.choice(seasons),
            "area_planted_ha": area[i],
            "quantity_harvested_ton": qty[i],
            "market_price": float(np.random.uniform(3000, 20000)),
            "status": np.random.choice(["berhasil", "berhasil", "berhasil", "gagal_sebagian", "gagal"]),
        })
    harvest_df = pd.DataFrame(rows)
    return farms_df, harvest_df

farms_df, harvest_df = build_toy_sets()
farms_df.head()

In [ ]:
# Pilih data training: preferensi hasil ETL (S3), fallback sintetis.
if farm_features is not None and len(farm_features) > 100:
    train_df = farm_features.copy()
    req = [c for c in ("farm_size_hectare", "total_activities", "crop_diversity") if c in train_df.columns]
    score = 0.1 * (1 / np.sqrt(train_df["farm_size_hectare"].clip(lower=0.5)))
    if "total_activities" in train_df.columns:
        score = score + 0.3 * (train_df["total_activities"] / train_df["total_activities"].max())
    if "crop_diversity" in train_df.columns:
        score = score + 0.2 * (train_df["crop_diversity"] / train_df["crop_diversity"].max())
    score = score + np.random.normal(0, 0.15, size=len(train_df))
    train_df["risk_label"] = (score > np.quantile(score, 0.55)).astype(int)
    train_df["risk_percentage"] = np.clip(score / score.max() * 100, 0, 100)
else:
    train_df = farms_df.copy()
print("Train rows:", len(train_df))
print(train_df[["farm_id", "risk_label", "risk_percentage"]].head())

## 1. Risk Model

Klasifikasi biner (gagal panen / tidak) dengan output `risk_percentage` dari `predict_proba`. Fitur: ukuran farm, total aktivitas, volume aktivitas, diversitas crop, statistik region.

In [ ]:
RISK_FEATURES = [
    "farm_size_hectare",
    "total_activities",
    "total_activity_volume",
    "avg_activity_volume",
    "crop_diversity",
    "region_avg_quantity",
]
RISK_TARGET = "risk_label"

risk_x = train_df[[c for c in RISK_FEATURES if c in train_df.columns]].values
risk_y = train_df[RISK_TARGET].values

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(risk_x, risk_y, test_size=0.2, random_state=42)

risk_model = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier(n_estimators=200, random_state=42)),
])
risk_model.fit(Xr_tr, yr_tr)

yr_pred = risk_model.predict(Xr_te)
yr_prob = risk_model.predict_proba(Xr_te)[:, 1]
print("--- Risk model metrics ---")
print(f"Accuracy: {accuracy_score(yr_te, yr_pred):.3f}")
print(f"F1      : {f1_score(yr_te, yr_pred):.3f}")
print(f"AUC     : {roc_auc_score(yr_te, yr_prob):.3f}")
print(classification_report(yr_te, yr_pred, zero_division=0))

## 2. Yield Model

Regresi (time-series) untuk forecast volume panen per crop. Fitur: `period_offset` + komponen musiman (sin/cos). Model menghasilkan `forecast_quantity_ton` untuk N periode ke depan.

In [ ]:
def build_yield_dataset(harvest: pd.DataFrame, stats: pd.DataFrame | None) -> pd.DataFrame:
    """Transform harvest history -> urutan per crop dengan period_offset."""
    if harvest is not None and len(harvest) > 0 and "quantity_harvested_ton" in harvest.columns:
        rows = []
        for cid, g in harvest.groupby("crop_id"):
            g = g.sort_values("season").reset_index(drop=True)
            for i, r in enumerate(g.itertuples(index=False)):
                qty = float(getattr(r, "quantity_harvested_ton", 0) or 0)
                rows.append({"crop_id": cid, "period_offset": i, "forecast_quantity_ton": qty})
        return pd.DataFrame(rows)

    stats_use = stats if (stats is not None and len(stats) > 50) else crop_stats_local
    rows = []
    for r in stats_use.itertuples(index=False):
        base = float(getattr(r, "avg_yield", 1.0) or 1.0)
        for off in range(12):
            seasonality = 1 + 0.2 * np.sin(2 * np.pi * off / 6)
            rows.append({
                "crop_id": str(getattr(r, "crop_id")),
                "period_offset": off,
                "forecast_quantity_ton": float(max(base * seasonality + np.random.normal(0, 5), 0.1)),
            })
    return pd.DataFrame(rows)

# Fallback stats lokal bila crop_stats tidak tersedia dari S3
if crop_stats is not None and len(crop_stats) > 50:
    crop_stats_local = crop_stats.copy()
    if "avg_yield" not in crop_stats_local.columns:
        crop_stats_local["avg_yield"] = crop_stats_local.get("avg_yield_ton_per_ha", 1.0)
else:
    crop_stats_local = harvest_df.assign(avg_yield=harvest_df["quantity_harvested_ton"] / harvest_df["area_planted_ha"])

harvest_src = harvest_df if (farm_features is None or crop_stats is None) else None
yld = build_yield_dataset(harvest_src, crop_stats_local)
print("Yield dataset rows:", len(yld))
yld.head()

In [ ]:
def add_seasonal(offset: np.ndarray) -> np.ndarray:
    off = np.asarray(offset, dtype=float)
    return np.column_stack([off, np.sin(2 * np.pi * off / 12), np.cos(2 * np.pi * off / 12)])

yld = yld.copy()
yld["forecast_quantity_ton"] = np.nan_to_num(
    yld["forecast_quantity_ton"].replace([np.inf, -np.inf], np.nan),
    nan=yld["forecast_quantity_ton"].median(),
)

fy = yld["forecast_quantity_ton"].values.astype(float)
fx = add_seasonal(yld["period_offset"].values)

Fx_tr, Fx_te, fy_tr, fy_te = train_test_split(fx, fy, test_size=0.2, random_state=42)

yield_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42, max_depth=12)),
])
yield_model.fit(Fx_tr, fy_tr)

fy_pred = yield_model.predict(Fx_te)
print("--- Yield model metrics ---")
print(f"MAE : {mean_absolute_error(fy_te, fy_pred):.2f}")
print(f"RMSE: {root_mean_squared_error(fy_te, fy_pred):.2f}")

## 3. Simpan & Upload Model ke S3

- `models/risk_model.pkl`   → dibaca `lambda_risk_prediction` (berisi `feature_cols` + model)
- `models/yield_model.pkl`  → dibaca `lambda_yield_forecasting` (berisi `feature_cols` + model)

In [ ]:
def upload_model(obj, key: str) -> None:
    s3 = boto3.client("s3")
    body = pickle.dumps(obj)
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=body)
    print(f"Uploaded s3://{S3_BUCKET}/{key} ({len(body)} bytes)")

risk_payload = {
    "feature_cols": [c for c in RISK_FEATURES if c in train_df.columns],
    "model": risk_model,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "accuracy": float(accuracy_score(yr_te, yr_pred)),
        "f1": float(f1_score(yr_te, yr_pred)),
        "auc": float(roc_auc_score(yr_te, yr_prob)),
    },
}

yield_payload = {
    "feature_cols": ["period_offset", "sin_t", "cos_t"],
    "model": yield_model,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "mae": float(mean_absolute_error(fy_te, fy_pred)),
        "rmse": float(root_mean_squared_error(fy_te, fy_pred)),
    },
}

upload_model(risk_payload, f"{MODEL_PREFIX}risk_model.pkl")
upload_model(yield_payload, f"{MODEL_PREFIX}yield_model.pkl")
print("Done. Kedua model tersimpan (atau di-upload) ke S3.")

## Cara pakai di Lambda

Lambda membaca dict `{feature_cols, model}` lalu mengekstrak fitur dari DynamoDB sesuai urutan `feature_cols`.

```python
payload = pickle.loads(s3.get_object(Bucket=BUCKET, Key=KEY)["Body"].read())
model = payload["model"]
# risk: fitur dari FARMS_TABLE + CROP_TABLE
proba = model.predict_proba([features])[0][1]   # risk_percentage = proba * 100
# yield: fitur [period_offset, sin_t, cos_t]
qty = model.predict([features])
```